## Setup

In [ ]:
!pip install -q --upgrade trl bitsandbytes accelerate peft datasets transformers evaluate rouge_score nltk huggingface_hub torchao


## Authentication

In [ ]:
import os, torch
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
login(hf_token)
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')


## 1 · Dataset Preparation

In [ ]:
import os, random
from collections import Counter, defaultdict
from datasets import load_dataset, ClassLabel

LABEL_COLUMN      = 'label'
CONTEXT_FIELD     = 'claim'        # used in fine-tuning prompt
SAMPLES_PER_LABEL = 100            # 100/label × 5 labels = 500 total
SEED              = 42
DATA_DIR          = '/kaggle/working/data_splits'
HF_DATASET_REPO   = 'Novaspree/factify_5K_enriched'

random.seed(SEED)
os.makedirs(DATA_DIR, exist_ok=True)

print('[1] Loading fixed splits...')
try:
    forget_raw_ds = load_dataset(HF_DATASET_REPO,
                                 data_files='forget/forget_set_fixed.json', split='train')
    retain_raw_ds = load_dataset(HF_DATASET_REPO,
                                 data_files='retain/retain_set_fixed.json', split='train')
    print('  Loaded from forget/ retain/ subdirectories')
except Exception:
    forget_raw_ds = load_dataset(HF_DATASET_REPO,
                                 data_files='forget_set_fixed.json', split='train')
    retain_raw_ds = load_dataset(HF_DATASET_REPO,
                                 data_files='retain_set_fixed.json', split='train')
    print('  Loaded from repo root')

print(f'Full sizes  forget={len(forget_raw_ds)}  retain={len(retain_raw_ds)}')

def norm_label(x):
    return {LABEL_COLUMN: str(x[LABEL_COLUMN])}
forget_raw_ds = forget_raw_ds.map(norm_label)
retain_raw_ds = retain_raw_ds.map(norm_label)

all_labels = sorted(set(forget_raw_ds.unique(LABEL_COLUMN) +
                        retain_raw_ds.unique(LABEL_COLUMN)))
print(f'Labels ({len(all_labels)}): {all_labels}')

def check_coverage(ds, n_per_label, name):
    counts = Counter(str(x) for x in ds[LABEL_COLUMN])
    short  = {lbl: cnt for lbl, cnt in counts.items() if cnt < n_per_label}
    if short:
        print(f'  WARNING [{name}]: labels short of {n_per_label}: {short}')
    else:
        print(f'  [{name}] all labels >= {n_per_label} samples -- OK')

check_coverage(forget_raw_ds, SAMPLES_PER_LABEL, 'forget')
check_coverage(retain_raw_ds, SAMPLES_PER_LABEL, 'retain')

def stratified_subsample(hf_ds, n_per_label, seed=42):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for i, lbl in enumerate(hf_ds[LABEL_COLUMN]):
        by_label[lbl].append(i)
    indices = []
    for lab in sorted(by_label):
        pool = list(by_label[lab])
        rng.shuffle(pool)
        indices.extend(pool[:n_per_label])
    rng.shuffle(indices)
    return hf_ds.select(indices)

forget_sub = stratified_subsample(forget_raw_ds, SAMPLES_PER_LABEL)
retain_sub = stratified_subsample(retain_raw_ds, SAMPLES_PER_LABEL)
print(f'Subsampled  forget={len(forget_sub)}  retain={len(retain_sub)}')
print(f'  Forget dist: {dict(Counter(forget_sub[LABEL_COLUMN]))}')
print(f'  Retain dist: {dict(Counter(retain_sub[LABEL_COLUMN]))}')

# Embed claim context into question field to match fine-tuning format
def build_unlearn_sample(example):
    claim = example.get(CONTEXT_FIELD, '') or ''
    if claim.strip():
        user_text = f"Context: {claim}\n\nQuestion: {example['question']}"
    else:
        user_text = example['question']
    return {'question': user_text}

forget_sub = forget_sub.map(build_unlearn_sample)
retain_sub  = retain_sub.map(build_unlearn_sample)
print('Claim context embedded into question field.')
print(f'  Sample question: {forget_sub[0]["question"][:120]}')

def format_for_training(example):
    return {'text': (
        '<bos><start_of_turn>user\n'
        f"{example['question']}<end_of_turn>\n"
        '<start_of_turn>model\n'
        f"{example['answer']}<end_of_turn>"
    )}

def make_hf_dataset(ds):
    ds = ds.map(format_for_training)
    ds = ds.cast_column(LABEL_COLUMN, ClassLabel(names=all_labels))
    return ds

forget_dataset = make_hf_dataset(forget_sub)
retain_dataset = make_hf_dataset(retain_sub)

forget_dataset.save_to_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset.save_to_disk(f'{DATA_DIR}/retain_dataset')
print(f'Saved to {DATA_DIR}/  forget={len(forget_dataset)}  retain={len(retain_dataset)}')


## 2 · Adapter Download & Sanity Check

In [ ]:
import gc, re, shutil, os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import snapshot_download

MODEL_NAME        = 'google/gemma-3-4b-it'
ADAPTER_REPO      = 'Novaspree/factify-Gemma3-adapter-1'
LORA_ADAPTER_PATH = '/kaggle/working/lora_adapter'

gc.collect()
torch.cuda.empty_cache()

print(f'[2] Downloading adapter from {ADAPTER_REPO}...')
if os.path.exists(LORA_ADAPTER_PATH):
    shutil.rmtree(LORA_ADAPTER_PATH)
snapshot_download(repo_id=ADAPTER_REPO, local_dir=LORA_ADAPTER_PATH)
print(f'Adapter saved -> {LORA_ADAPTER_PATH}')

try:
    tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH)
    print('Tokenizer loaded from adapter repo')
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print('Tokenizer loaded from base model')

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<pad>'})
    print('Added <pad> token')
tokenizer.padding_side = 'right'
tokenizer.save_pretrained(LORA_ADAPTER_PATH)
print(f'Pad token: {tokenizer.pad_token!r}  id={tokenizer.pad_token_id}')

# ── Sanity-check: confirm LoRA layers 9-20 ──────────────────────────────────
print('\nSanity check -- loading PeftModel (bfloat16)...')
_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto'
)
_cur_vocab = _base.get_input_embeddings().weight.shape[0]
if len(tokenizer) != _cur_vocab:
    _base.resize_token_embeddings(len(tokenizer))
    print(f'Resized embeddings: {_cur_vocab} -> {len(tokenizer)}')

_peft = PeftModel.from_pretrained(_base, LORA_ADAPTER_PATH, is_trainable=False)
lora_layers = set()
for n, _ in _peft.named_modules():
    m = re.search(r'layers\.(\d+)', n)
    if m and ('lora_A' in n or 'lora_B' in n):
        lora_layers.add(int(m.group(1)))
if lora_layers:
    print(f'LoRA layer range: {min(lora_layers)}-{max(lora_layers)}  (expected 9-20)')
del _peft, _base
gc.collect(); torch.cuda.empty_cache()
print('Sanity check passed.')


## 3 · Utility Functions

> Standalone cell — run before the engine and any evaluation cell.

In [ ]:
# ============================================================
# Standalone utility functions
# Run this cell before the engine and any evaluation cell.
# Keeping them here means c-eval and c-rouge work even when
# run in a fresh kernel after loading the merged model.
# ============================================================
import re
import torch
import torch.nn as nn


def format_prompt_eval(question: str) -> str:
    """Gemma-3-it chat template (generation side).

    Mirrors tokenizer.apply_chat_template(..., add_generation_prompt=True).
    ALWAYS pair with add_special_tokens=False: Gemma's SentencePiece tokeniser
    prepends BOS automatically, and <bos> is already inside this string.
    """
    return (
        '<bos><start_of_turn>user\n'
        f'{question}<end_of_turn>\n'
        '<start_of_turn>model\n'
    )


def make_answer_only_labels(input_ids: torch.Tensor, prompt_len: int) -> torch.Tensor:
    labels = input_ids.clone()
    labels[:, :prompt_len] = -100
    return labels


def get_input_device(model) -> torch.device:
    return next(model.parameters()).device


_GARBAGE_TOKENS = [
    'kter', 'GOOG', 'sinstance', 'Achtung', 'ndef', 'endoftext', 'AAAA', 'BBBB',
]

def is_output_broken(text: str, severe_only: bool = False) -> bool:
    if not text or len(text.strip()) < 5:
        return True
    alnum = sum(c.isalnum() or c in ' .,!?-' for c in text)
    if alnum / max(1, len(text)) < 0.35:
        return True
    if severe_only:
        return False
    for tok in _GARBAGE_TOKENS:
        if tok in text:
            return True
    if text.count('?') > 3 and len(text) < 300:
        return True
    words = text.split()
    if words and sum(len(w) for w in words) / len(words) > 14:
        return True
    return False


def clean_output(text: str) -> str:
    text = re.sub(r'<(end_of_turn|start_of_turn|bos|eos)>.*$', '', text, flags=re.DOTALL)
    text = re.sub(r'\s*(CLIIIK|\?>|\];|\]|/\*|<!|\{\{|\}\}).*$', '', text, flags=re.DOTALL)
    return text.strip()


def check_weights_nan(named_lora_mods, label: str = '') -> tuple:
    """Scan LoRA weight tensors for NaN / Inf values.
    Returns (healthy: bool, n_nan: int, n_inf: int).
    """
    n_nan = n_inf = 0
    for _, mod in named_lora_mods:
        if isinstance(mod, nn.Linear):
            n_nan += int(torch.isnan(mod.weight.data).sum())
            n_inf += int(torch.isinf(mod.weight.data).sum())
    healthy = (n_nan == 0 and n_inf == 0)
    if label:
        status = '\u2705 healthy' if healthy else f'\u274c  NaN={n_nan:,}  Inf={n_inf:,}'
        print(f'  Weight health [{label}]: {status}')
    return healthy, n_nan, n_inf


print('Utility functions ready: format_prompt_eval, clean_output, get_input_device,')
print('                         is_output_broken, check_weights_nan.')


## 4 · RecursiveMAAT Engine (v9.4)

In [ ]:
import math, re, gc, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm


class RecursiveMAAT_v9:
    """
    RecursiveMAAT v9.4 -- Gemma-3-4B edition.
    """

    UNLEARN_MODULE_TYPES = ('down_proj', 'up_proj', 'q_proj', 'v_proj')
    REPAIR_MODULE_TYPES  = ('down_proj', 'up_proj', 'gate_proj',
                            'q_proj', 'k_proj', 'v_proj', 'o_proj')
    SVD_MLP_TYPES        = ('down_proj', 'up_proj', 'gate_proj')
    SVD_ATTN_TYPES       = ('o_proj',)
    HIDDEN_LAYER_IDX     = 20   # output of decoder layer 20 (LoRA end)
    LOSS_CLAMP_MAX       = 20.0 # clamp forget loss -- prevents log(~0)→-inf→NaN

    def __init__(
        self, model,
        mid_layer_start=9, mid_layer_end=20,
        learning_rate=1e-5, max_grad_norm=1.0,
        kl_temp=0.7,
        do_svd_prune=True, svd_prune_ratio=0.15,
        attn_prune_ratio=0.02, attn_prune_layer_start=14,
    ):
        self.model            = model
        self.lr               = learning_rate
        self.max_grad_norm    = max_grad_norm
        self.kl_temp          = kl_temp
        self.do_svd_prune     = do_svd_prune
        self.svd_prune_ratio  = svd_prune_ratio
        self.attn_prune_ratio = attn_prune_ratio
        self.attn_prune_ls    = attn_prune_layer_start
        self.mid_layer_start  = mid_layer_start
        self.mid_layer_end    = mid_layer_end
        self.forget_task_vec  = {}
        self._nan_skip_count  = 0   # BUG-2: track skipped NaN steps

        # ── Phase-1 targets (UNLEARN types, layers mid_start..mid_end) ───────
        self.lora_a_modules = []
        self.lora_b_modules = []
        for name, mod in model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m: continue
            if not (mid_layer_start <= int(m.group(1)) <= mid_layer_end): continue
            if not any(mt in name for mt in self.UNLEARN_MODULE_TYPES): continue
            if   'lora_A' in name: self.lora_a_modules.append((name, mod))
            elif 'lora_B' in name: self.lora_b_modules.append((name, mod))
        self.target_modules = self.lora_a_modules + self.lora_b_modules
        if not self.target_modules:
            raise RuntimeError('No lora_A/lora_B modules found. Pass a trainable PeftModel.')

        # ── o_proj for Phase 2b ───────────────────────────────────────────────
        self.attn_a_modules = []
        self.attn_b_modules = []
        for name, mod in model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m: continue
            layer = int(m.group(1))
            if not (attn_prune_layer_start <= layer <= mid_layer_end): continue
            if not any(mt in name for mt in self.SVD_ATTN_TYPES): continue
            if   'lora_A' in name: self.attn_a_modules.append((name, mod))
            elif 'lora_B' in name: self.attn_b_modules.append((name, mod))

        # ── BUG-5 FIX: cache ALL LoRA weights on GPU (no .cpu()) ─────────────
        # Previously stored as CPU tensors, causing 200k+ CPU↔GPU copies.
        # GPU-resident tensors make .to(device) a no-op on same-device modules.
        self.all_lora_mods = [
            (n, mod) for n, mod in model.named_modules()
            if isinstance(mod, nn.Linear) and ('lora_A' in n or 'lora_B' in n)
        ]
        self.finetuned_weights = {
            n: mod.weight.data.clone()   # stays on GPU -- BUG-5 FIX
            for n, mod in self.all_lora_mods
        }

        # ── Freeze all; unfreeze Phase-1 targets ─────────────────────────────
        for p in model.parameters():
            p.requires_grad = False
        for _, mod in self.target_modules:
            mod.weight.requires_grad = True

        # ── BUG-3 FIX: single persistent Phase-1 optimizer ───────────────────
        # Previously a new Adam instance was created for every forget sample
        # (500×), losing all accumulated moment estimates between samples.
        self.phase1_opt = torch.optim.Adam(
            [m.weight for _, m in self.target_modules],
            lr=self.lr, betas=(0.9, 0.999), eps=1e-8,
        )

        n_unlearn = sum(m.weight.numel() for _, m in self.target_modules)
        n_total   = sum(p.numel() for p in model.parameters())
        print('RecursiveMAAT v9.4 [Gemma-3-4B] ready')
        print(f'  Phase-1 targets  : {len(self.target_modules)} modules '
              f'({len(self.lora_a_modules)}A + {len(self.lora_b_modules)}B)')
        print(f'  Trainable params : {n_unlearn:,} / {n_total:,} '
              f'({100*n_unlearn/n_total:.3f}%)')
        print(f'  lr={learning_rate}  kl_temp={kl_temp}  '
              f'mlp_svd={svd_prune_ratio:.0%}  loss_clamp={self.LOSS_CLAMP_MAX}')

    # ── Diagnostic helpers ────────────────────────────────────────────────────

    def check_weights_healthy(self, label: str = '') -> tuple:
        """Scan all LoRA tensors for NaN/Inf. Returns (healthy, n_nan, n_inf)."""
        return check_weights_nan(self.all_lora_mods, label)

    def recover_nan_weights(self) -> int:
        """Restore any NaN/Inf LoRA modules from the finetuned checkpoint.
        Returns the number of modules that were restored.
        """
        restored = 0
        with torch.no_grad():
            for name, mod in self.all_lora_mods:
                if not torch.isfinite(mod.weight.data).all():
                    fw = self.finetuned_weights.get(name)
                    if fw is not None:
                        mod.weight.data.copy_(fw.to(mod.weight.device))
                        restored += 1
        if restored:
            print(f'  Restored {restored} NaN/Inf modules from finetuned checkpoint.')
        return restored

    # ── Encoding helper ───────────────────────────────────────────────────────

    def _encode(self, question, answer, tokenizer, device):
        """Encode prompt+answer with answer-only labels.

        BUG-6 FIX: appends '<end_of_turn>' to match the fine-tuning format.
        Previously omitted, so the EOS prediction was excluded from the
        unlearning gradient signal.

        add_special_tokens=False: format_prompt_eval() already includes <bos>.
        token_type_ids=zeros: Gemma3's multimodal forward router requires this
        when pixel_values is None to avoid 'token_type_ids required' errors.
        """
        prompt = format_prompt_eval(question)
        plen   = tokenizer(
            prompt, return_tensors='pt', add_special_tokens=False
        ).input_ids.shape[1]
        # BUG-6 FIX: include <end_of_turn> so EOS is part of unlearning signal
        full_seq = prompt + answer + '<end_of_turn>'
        enc = tokenizer(
            full_seq, return_tensors='pt',
            truncation=True, max_length=512,
            add_special_tokens=False,
        ).to(device)
        enc['token_type_ids'] = torch.zeros_like(enc['input_ids'])
        return enc, make_answer_only_labels(enc['input_ids'], plen)

    # ── Reference forward passes ──────────────────────────────────────────────

    def _get_finetuned_logits(self, enc):
        """Forward pass with finetuned weights (KL anchor for Phase 1)."""
        self.model.eval()
        saved = {n: mod.weight.data.clone() for n, mod in self.all_lora_mods}
        for n, mod in self.all_lora_mods:
            fw = self.finetuned_weights.get(n)
            if fw is not None:
                mod.weight.data.copy_(fw.to(mod.weight.device))
        with torch.no_grad():
            logits = self.model(**enc).logits.float().detach()
        for n, mod in self.all_lora_mods:
            mod.weight.data.copy_(saved[n])
        del saved
        return logits

    def _get_finetuned_outputs(self, enc):
        """Forward pass with finetuned weights returning logits + hidden state."""
        self.model.eval()
        saved = {n: mod.weight.data.clone() for n, mod in self.all_lora_mods}
        for n, mod in self.all_lora_mods:
            fw = self.finetuned_weights.get(n)
            if fw is not None:
                mod.weight.data.copy_(fw.to(mod.weight.device))
        with torch.no_grad():
            out    = self.model(**enc, output_hidden_states=True)
            logits = out.logits.float().detach()
            hs     = out.hidden_states[self.HIDDEN_LAYER_IDX + 1]
            hs_ref = hs.float().mean(dim=1).detach()
            del out
        for n, mod in self.all_lora_mods:
            mod.weight.data.copy_(saved[n])
        del saved
        return logits, hs_ref

    # ── Phase 1: GradProject unlearning ──────────────────────────────────────

    def unlearn_step(self, forget_question, forget_answer,
                     retain_pool, retain_start_idx, tokenizer, steps=2):
        """Gradient-projected ascent on forget data while protecting retain.

        BUG-2 FIX: non-finite loss / gradient tensors are detected and the
                   step is cleanly skipped instead of propagating NaN.
        BUG-3 FIX: uses self.phase1_opt (persistent across samples) instead
                   of a fresh Adam instance per call.
        BUG-4 FIX: GradProject is conditional on dot(g_f, g_r) > 0.
                   When the gradients do not conflict the full forget-ascent
                   gradient is used; previously always projected, attenuating
                   the unlearning signal unnecessarily.
        """
        self.model.train()
        device = get_input_device(self.model)
        forget_enc, forget_labels = self._encode(
            forget_question, forget_answer, tokenizer, device)

        for step in range(steps):
            rs = retain_pool[(retain_start_idx + step) % len(retain_pool)]
            retain_enc, _ = self._encode(rs['question'], rs['answer'], tokenizer, device)

            # ── Forget gradient (ascent direction) ───────────────────────────
            self.phase1_opt.zero_grad()
            forget_loss = self.model(**forget_enc, labels=forget_labels).loss

            # BUG-2 FIX: skip step if loss is non-finite
            if not torch.isfinite(forget_loss):
                self._nan_skip_count += 1
                del retain_enc
                continue

            # Extra safety: clamp loss to prevent log(~0) -> -inf -> NaN grad
            forget_loss = forget_loss.clamp(max=self.LOSS_CLAMP_MAX)
            forget_loss.backward()

            # Validate gradients before saving (BUG-2 FIX, second guard)
            f_grads = {}
            bad_grad = False
            for n, m in self.target_modules:
                g = m.weight.grad
                if g is None:
                    continue
                if not torch.isfinite(g).all():
                    bad_grad = True
                    break
                f_grads[n] = g.clone()

            if bad_grad:
                self.phase1_opt.zero_grad()
                self._nan_skip_count += 1
                del retain_enc
                continue

            # ── Retain KL gradient ───────────────────────────────────────────
            self.phase1_opt.zero_grad()
            ref_logits = self._get_finetuned_logits(retain_enc)
            ref_probs  = F.softmax(ref_logits / self.kl_temp, dim=-1)
            del ref_logits
            self.model.train()
            kl_loss = F.kl_div(
                F.log_softmax(
                    self.model(**retain_enc).logits.float() / self.kl_temp,
                    dim=-1),
                ref_probs, reduction='batchmean')
            del ref_probs
            kl_loss.backward()

            # ── BUG-4 FIX: conditional GradProject ──────────────────────────
            # Project forget-ascent gradient orthogonal to retain gradient
            # ONLY when they conflict (dot > 0). Previously always projected,
            # unnecessarily reducing the unlearning signal.
            with torch.no_grad():
                for name, mod in self.target_modules:
                    g_r = mod.weight.grad
                    if g_r is None:
                        continue
                    g_f = f_grads.get(name)
                    if g_f is None:
                        mod.weight.grad.zero_()
                        continue
                    g_f = g_f.to(g_r.device)
                    dot = (g_f * g_r).sum()
                    # BUG-4 FIX: only orthogonalise when gradients conflict
                    if dot > 0:
                        g_f_proj = g_f - (dot / (g_r * g_r).sum().clamp(1e-12)) * g_r
                    else:
                        g_f_proj = g_f   # no conflict: full forget ascent
                    mod.weight.grad.copy_(-g_f_proj)

            torch.nn.utils.clip_grad_norm_(
                [m.weight for _, m in self.target_modules], self.max_grad_norm)
            self.phase1_opt.step()
            del f_grads, retain_enc

        torch.cuda.empty_cache()

    # ── Shared prune helper ───────────────────────────────────────────────────

    def _score_and_prune_lora(self, forget_dataset, tokenizer,
                               a_mods, b_mods, prune_ratio,
                               n_score_samples, desc):
        device = get_input_device(self.model)
        self.model.eval()

        def bkey(n): return re.sub(r'\.lora_[AB].*', '', n)
        a_by = {bkey(n): (n, m) for n, m in a_mods}
        b_by = {bkey(n): (n, m) for n, m in b_mods}
        keys = set(a_by) & set(b_by)

        sig = {}
        for i in tqdm(range(min(n_score_samples, len(forget_dataset))), desc=desc):
            s = forget_dataset[i]
            enc, lbl = self._encode(s['question'], s['answer'], tokenizer, device)
            self.model.zero_grad()
            loss = self.model(**enc, labels=lbl).loss
            if torch.isfinite(loss):
                loss.backward()
            with torch.no_grad():
                for k in keys:
                    _, mb = b_by[k]
                    if mb.weight.grad is None: continue
                    sig[k] = sig.get(k, 0) + mb.weight.grad.float().norm(dim=0).cpu()
        self.model.zero_grad(); gc.collect(); torch.cuda.empty_cache()

        zeroed = 0
        for k in keys:
            _, ma = a_by[k]; _, mb = b_by[k]
            rank = ma.weight.shape[0]
            n_p  = max(1, int(prune_ratio * rank))
            dims = sig.get(k, torch.ones(rank)).argsort(descending=True)[:n_p]
            with torch.no_grad():
                mb.weight.data[:, dims] = 0.0
                ma.weight.data[dims, :] = 0.0
            zeroed += n_p
        return zeroed

    # ── Phase 2a: MLP SVD pruning ─────────────────────────────────────────────

    def svd_prune(self, forget_dataset, tokenizer, n_score_samples=20):
        if not self.do_svd_prune: return
        mlp_a = [(n, m) for n, m in self.lora_a_modules
                 if any(t in n for t in self.SVD_MLP_TYPES)]
        mlp_b = [(n, m) for n, m in self.lora_b_modules
                 if any(t in n for t in self.SVD_MLP_TYPES)]
        n_pairs = min(len(mlp_a), len(mlp_b))
        print(f'\nPhase 2a -- MLP SVD ({self.svd_prune_ratio:.0%}, {n_pairs} pairs)...')
        zeroed = self._score_and_prune_lora(
            forget_dataset, tokenizer,
            mlp_a, mlp_b, self.svd_prune_ratio,
            n_score_samples, desc='Scoring MLP dims',
        )
        print(f'  Zeroed {zeroed} MLP rank dims.')

    # ── Phase 2b: o_proj micro-prune ──────────────────────────────────────────

    def attn_micro_prune(self, forget_dataset, tokenizer, n_score_samples=20):
        if not self.attn_a_modules:
            print('  No o_proj lora modules found -- skipping.'); return
        print(f'\nPhase 2b -- o_proj micro-prune ({self.attn_prune_ratio:.0%})...')
        zeroed = self._score_and_prune_lora(
            forget_dataset, tokenizer,
            self.attn_a_modules, self.attn_b_modules, self.attn_prune_ratio,
            n_score_samples, desc='Scoring o_proj dims',
        )
        print(f'  Zeroed {zeroed} o_proj rank dims.')

    # ── Phase 2.5A: forget task vectors ───────────────────────────────────────

    def compute_forget_task_vector(self, forget_dataset, tokenizer,
                                   n_score_samples=20):
        print(f'\nPhase 2.5A -- Computing forget task vectors '
              f'({n_score_samples} scoring samples)...')
        device = get_input_device(self.model)
        self.model.eval()

        b_mods = {}
        for name, mod in self.model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m or not (self.mid_layer_start <= int(m.group(1))
                             <= self.mid_layer_end): continue
            if not any(mt in name for mt in self.REPAIR_MODULE_TYPES): continue
            if 'lora_B' in name:
                b_mods[name] = mod

        print(f'  {len(b_mods)} lora_B targets (all 7 types, '
              f'layers {self.mid_layer_start}-{self.mid_layer_end})')

        saved_req_grad = {name: mod.weight.requires_grad
                          for name, mod in b_mods.items()}
        for mod in b_mods.values():
            mod.weight.requires_grad_(True)

        sig = {}
        try:
            for i in tqdm(range(min(n_score_samples, len(forget_dataset))),
                          desc='Scoring forget dims (all lora_B)'):
                s = forget_dataset[i]
                enc, lbl = self._encode(s['question'], s['answer'], tokenizer, device)
                self.model.zero_grad()
                loss = self.model(**enc, labels=lbl).loss
                if torch.isfinite(loss):
                    loss.backward()
                with torch.no_grad():
                    for name, mod in b_mods.items():
                        if mod.weight.grad is None: continue
                        sig[name] = (sig.get(name, 0)
                                     + mod.weight.grad.float().norm(dim=0).cpu())
        finally:
            for name, mod in b_mods.items():
                mod.weight.requires_grad_(saved_req_grad[name])
            self.model.zero_grad()
            gc.collect(); torch.cuda.empty_cache()

        self.forget_task_vec = {}
        total_masked = 0
        for name, mod in b_mods.items():
            scores = sig.get(name)
            if scores is None: continue
            rank     = mod.weight.shape[1]
            n_top    = max(1, rank // 2)
            top_dims = scores.argsort(descending=True)[:n_top]
            mask     = torch.zeros(rank, dtype=torch.float32)
            mask[top_dims] = 1.0
            ft_w = self.finetuned_weights.get(name)
            if ft_w is None: continue
            self.forget_task_vec[name] = ft_w.float() * mask.unsqueeze(0).to(ft_w.device)
            total_masked += n_top

        n_mods  = len(self.forget_task_vec)
        per_mod = total_masked // max(1, n_mods)
        print(f'  Task vectors built for {n_mods} modules | '
              f'{total_masked} total forget-coded dims (~{per_mod} per module)')

    # ── Phase 2.5B: task vector negation ──────────────────────────────────────

    def apply_task_vector_negation(self, alpha: float = 1.0):
        if not self.forget_task_vec:
            print('  WARNING: forget_task_vec empty. Call compute_forget_task_vector first.')
            return
        print(f'\nPhase 2.5B -- Task vector negation (alpha={alpha})...')
        n_applied = 0
        with torch.no_grad():
            for name, mod in self.model.named_modules():
                if not isinstance(mod, nn.Linear): continue
                tv = self.forget_task_vec.get(name)
                if tv is None: continue
                mod.weight.data -= alpha * tv.to(mod.weight.device)
                n_applied += 1
        print(f'  Subtracted alpha*forget_task_vec from {n_applied} lora_B modules.')

    # ── Phase 3: Forget-aware hybrid repair ───────────────────────────────────

    def retain_repair(
        self, retain_dataset, tokenizer,
        forget_dataset=None,
        n_steps=150, repair_lr=8e-5,
        kl_weight=0.60, hs_weight=0.25,
        forget_reg_weight=0.10,
        tv_weight=0.05,
        repair_kl_temp=2.0, final_lr=5e-6,
    ):
        """Forget-aware hybrid repair with TV regulariser (v9.4).

        Loss = 0.60*KL + 0.25*HS + 0.10*(-entropy_forget) + 0.05*TV_reg

        Additional improvement: non-finite total loss triggers a skip with a
        warning instead of propagating NaN through weight updates.
        """
        print(f'\nPhase 3 -- Forget-aware hybrid repair (v9.4)  '
              f'kl={kl_weight} hs={hs_weight} '
              f'freg={forget_reg_weight} tv={tv_weight}  '
              f'kl_temp={repair_kl_temp}  '
              f'lr {repair_lr:.0e}->{final_lr:.0e}  steps={n_steps}')
        device = get_input_device(self.model)
        self.model.train()

        repair_mods = []
        for name, mod in self.model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m or not (self.mid_layer_start <= int(m.group(1))
                             <= self.mid_layer_end): continue
            if not any(mt in name for mt in self.REPAIR_MODULE_TYPES): continue
            if 'lora_A' in name or 'lora_B' in name:
                mod.weight.requires_grad = True
                repair_mods.append((name, mod))
        print(f'  Repair modules: {len(repair_mods)}')

        has_tv = bool(self.forget_task_vec) and tv_weight > 0
        print(f'  TV reg: {"active (" + str(len(self.forget_task_vec)) + " modules)" if has_tv else "inactive"}')

        heavy_set = set(self.UNLEARN_MODULE_TYPES)
        heavy_p = [m.weight for n, m in repair_mods if any(t in n for t in heavy_set)]
        light_p = [m.weight for n, m in repair_mods if not any(t in n for t in heavy_set)]
        opt = torch.optim.Adam([
            {'params': heavy_p, 'lr': repair_lr},
            {'params': light_p, 'lr': repair_lr * 0.3},
        ])

        samples   = [retain_dataset[i] for i in range(len(retain_dataset))]
        random.shuffle(samples)
        f_samples = None
        if forget_dataset is not None and forget_reg_weight > 0:
            f_samples = [forget_dataset[i] for i in range(len(forget_dataset))]
            random.shuffle(f_samples)
            print(f'  Forget regulariser: {len(f_samples)} samples')

        nan_repair_skips = 0
        for step in range(n_steps):
            cos = 0.5 * (1.0 + math.cos(math.pi * step / n_steps))
            opt.param_groups[0]['lr'] = final_lr + (repair_lr       - final_lr) * cos
            opt.param_groups[1]['lr'] = final_lr + (repair_lr * 0.3 - final_lr) * cos

            sample = samples[step % len(samples)]
            enc, _ = self._encode(sample['question'], sample['answer'], tokenizer, device)
            opt.zero_grad()

            ref_logits, ref_hs = self._get_finetuned_outputs(enc)
            ref_probs = F.softmax(ref_logits / repair_kl_temp, dim=-1)
            del ref_logits

            self.model.train()
            out          = self.model(**enc, output_hidden_states=True)
            model_logits = out.logits.float()
            curr_hs      = out.hidden_states[self.HIDDEN_LAYER_IDX + 1].float().mean(dim=1)
            del out

            kl_loss = F.kl_div(
                F.log_softmax(model_logits / repair_kl_temp, dim=-1),
                ref_probs, reduction='batchmean')
            hs_loss = 1.0 - F.cosine_similarity(curr_hs, ref_hs, dim=-1).mean()

            forget_reg_loss = torch.tensor(0.0, device=device)
            if f_samples is not None:
                fs = f_samples[step % len(f_samples)]
                f_enc, f_labels = self._encode(fs['question'], fs['answer'],
                                               tokenizer, device)
                f_logits = self.model(**f_enc).logits.float()
                f_mask   = (f_labels[0] != -100)
                if f_mask.any():
                    f_ans = f_logits[0][f_mask]
                    f_lp  = F.log_softmax(f_ans, dim=-1)
                    f_ent = -(f_lp.exp() * f_lp).sum(dim=-1).mean()
                    forget_reg_loss = -f_ent
                del f_logits

            tv_reg_loss = torch.tensor(0.0, device=device)
            if has_tv:
                tv_sims = []
                for name, mod in self.model.named_modules():
                    if not isinstance(mod, nn.Linear): continue
                    tv = self.forget_task_vec.get(name)
                    if tv is None: continue
                    tv_dev    = tv.to(mod.weight.device)
                    curr_flat = mod.weight.float().view(1, -1)
                    tv_flat   = tv_dev.view(1, -1)
                    sim = F.cosine_similarity(curr_flat, tv_flat, dim=1)
                    tv_sims.append(sim.clamp(min=0.0))
                if tv_sims:
                    tv_reg_loss = torch.cat(tv_sims).mean()

            total = (kl_weight         * kl_loss
                   + hs_weight         * hs_loss
                   + forget_reg_weight * forget_reg_loss
                   + tv_weight         * tv_reg_loss)
            del ref_probs, model_logits, curr_hs, ref_hs, enc

            # Phase-3 NaN guard: skip non-finite steps instead of corrupting weights
            if not torch.isfinite(total):
                opt.zero_grad()
                nan_repair_skips += 1
                if nan_repair_skips <= 5:
                    print(f'  WARNING: non-finite loss at step {step+1} -- skipping.')
                continue

            total.backward()
            torch.nn.utils.clip_grad_norm_(
                [m.weight for _, m in repair_mods], self.max_grad_norm)
            opt.step()

            if (step + 1) % 25 == 0:
                torch.cuda.empty_cache()
                freg_val = forget_reg_loss.item() if isinstance(forget_reg_loss, torch.Tensor) else 0.0
                tv_val   = tv_reg_loss.item()   if isinstance(tv_reg_loss,   torch.Tensor) else 0.0
                print(f'  step {step+1:3d}/{n_steps}  '
                      f'kl={kl_loss.item():.4f}  hs={hs_loss.item():.4f}  '
                      f'freg={freg_val:.4f}  tv={tv_val:.4f}  '
                      f'total={total.item():.4f}  '
                      f'lr={opt.param_groups[0]["lr"]:.2e}')

        if nan_repair_skips:
            print(f'  Phase-3 NaN skips: {nan_repair_skips}/{n_steps}')
        print('Hybrid repair complete.')


print('RecursiveMAAT v9.4 (Gemma-3-4B) class loaded.')


## 5 · Unlearning Run

> Includes pre-unlearning generation check and per-phase weight health diagnostics.

In [ ]:
import os, shutil, gc, random, ast
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk
import evaluate as hf_evaluate

MODEL_NAME        = 'google/gemma-3-4b-it'
LORA_ADAPTER_PATH = '/kaggle/working/lora_adapter'
UNLEARNED_ADAPTER = '/kaggle/working/lora_adapter_unlearned'
DATA_DIR          = '/kaggle/working/data_splits'

# ── Hyperparameters ────────────────────────────────────────────────────────
MID_LAYER_START  = 9
MID_LAYER_END    = 20
LEARNING_RATE    = 4e-6
MAX_GRAD_NORM    = 1.0
UNLEARN_STEPS    = 2
KL_TEMP          = 0.7
USE_REPHRASES    = False
MAX_REPHRASES    = 1     # stride only; rephrases disabled above
REPHRASE_STEPS   = 1

DO_SVD_PRUNE      = True
SVD_PRUNE_RATIO   = 0.02
SVD_SCORE_SAMPLES = 60

DO_ATTN_PRUNE = False

DO_TASK_VEC      = True
TV_SCORE_SAMPLES = 60
TV_ALPHA         = 0.05

DO_RETAIN_REPAIR  = True
REPAIR_STEPS      = 300
REPAIR_LR         = 8e-5
KL_WEIGHT         = 0.60
HS_WEIGHT         = 0.25
FORGET_REG_WEIGHT = 0.10
TV_WEIGHT         = 0.05
REPAIR_KL_TEMP    = 2.0
FINAL_LR          = 5e-6

PRE_CHECK_SAMPLES = 5   # qualitative generation samples per split before unlearning
# ──────────────────────────────────────────────────────────────────────────────

gc.collect(); torch.cuda.empty_cache()
random.seed(42)

forget_dataset = load_from_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset = load_from_disk(f'{DATA_DIR}/retain_dataset')
retain_list    = [retain_dataset[i] for i in range(len(retain_dataset))]
random.shuffle(retain_list)
print(f'Forget: {len(forget_dataset)} | Retain: {len(retain_dataset)}')
assert len(forget_dataset) == 500
assert len(retain_dataset) == 500

tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<pad>'})
    tokenizer.save_pretrained(LORA_ADAPTER_PATH)
print(f'Pad token: {tokenizer.pad_token!r}  id={tokenizer.pad_token_id}')

# ── BUG-1 FIX: load in bfloat16 ───────────────────────────────────────────
# Fine-tuning used bf16=True; loading in float16 caused fp16 overflow during
# gradient ascent (max fp16 = 65504 vs bf16 = ~3.4e38), producing NaN weights
# and broken Phase-3 repair (all-NaN losses observed in v9.3).
print('Loading base model + LoRA adapter (bfloat16, no quantisation)...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto'   # BUG-1 FIX
)
_cur_vocab = base_model.get_input_embeddings().weight.shape[0]
if len(tokenizer) != _cur_vocab:
    base_model.resize_token_embeddings(len(tokenizer))
    print(f'  Resized embeddings: {_cur_vocab} -> {len(tokenizer)}')

model = PeftModel.from_pretrained(
    base_model, LORA_ADAPTER_PATH, is_trainable=True
)
model.eval()

# Gemma EOS ids for generation checks
_eos_ids = list(set(filter(lambda x: x is not None and x >= 0, [
    tokenizer.convert_tokens_to_ids('<end_of_turn>'),
    tokenizer.eos_token_id,
])))
print(f'EOS token ids: {_eos_ids}')

# ════════════════════════════════════════════════════════════════════════════
# PRE-UNLEARNING GENERATION CHECK
# Generate answers from the finetuned model before any modification.
# This establishes a qualitative baseline and verifies the adapter is intact.
# ════════════════════════════════════════════════════════════════════════════
print('\n' + '='*70)
print('PRE-UNLEARNING GENERATION CHECK')
print('='*70)

def _quick_gen(question: str, max_new: int = 80) -> str:
    prompt = format_prompt_eval(question)
    inp = tokenizer(prompt, return_tensors='pt', truncation=True,
                    max_length=400, add_special_tokens=False).to(
        get_input_device(model))
    inp['token_type_ids'] = torch.zeros_like(inp['input_ids'])
    with torch.no_grad():
        gen = model.generate(
            **inp, max_new_tokens=max_new, do_sample=False,
            repetition_penalty=1.3,
            eos_token_id=_eos_ids, pad_token_id=tokenizer.pad_token_id)
    raw = tokenizer.decode(gen[0][inp['input_ids'].shape[1]:],
                           skip_special_tokens=True).strip()
    return clean_output(raw)

print(f'\n[ Forget set – model should answer these ({PRE_CHECK_SAMPLES} samples) ]')
for _i in range(PRE_CHECK_SAMPLES):
    _s = forget_dataset[_i]
    _pred = _quick_gen(_s['question'])
    _ok = not is_output_broken(_pred)
    print(f'  #{_i+1}  {"✅" if _ok else "⚠️ "}')
    print(f'    Q   : {_s["question"][:110]}')
    print(f'    GT  : {_s["answer"][:110]}')
    print(f'    Pred: {_pred[:110]}\n')

print(f'[ Retain set – model should answer these ({PRE_CHECK_SAMPLES} samples) ]')
retain_ok = 0
for _i in range(PRE_CHECK_SAMPLES):
    _s = retain_dataset[_i]
    _pred = _quick_gen(_s['question'])
    _ok = not is_output_broken(_pred)
    if _ok: retain_ok += 1
    print(f'  #{_i+1}  {"✅" if _ok else "⚠️ "}')
    print(f'    Q   : {_s["question"][:110]}')
    print(f'    GT  : {_s["answer"][:110]}')
    print(f'    Pred: {_pred[:110]}\n')

print(f'Pre-check retain coherence: {retain_ok}/{PRE_CHECK_SAMPLES}')
if retain_ok < PRE_CHECK_SAMPLES // 2:
    print('WARNING: adapter may not be loading correctly. Check LORA_ADAPTER_PATH.')
else:
    print('Adapter looks healthy -- proceeding to unlearning.')
print('='*70 + '\n')

# ════════════════════════════════════════════════════════════════════════════
# UNLEARNING
# ════════════════════════════════════════════════════════════════════════════
maat = RecursiveMAAT_v9(
    model,
    mid_layer_start        = MID_LAYER_START,
    mid_layer_end          = MID_LAYER_END,
    learning_rate          = LEARNING_RATE,
    max_grad_norm          = MAX_GRAD_NORM,
    kl_temp                = KL_TEMP,
    do_svd_prune           = DO_SVD_PRUNE,
    svd_prune_ratio        = SVD_PRUNE_RATIO,
    attn_prune_ratio       = 0.02,
    attn_prune_layer_start = 14,
)

def get_rephrases(sample):
    raw = sample.get('rephrases', [])
    if isinstance(raw, list):
        return [r for r in raw if r and str(r).strip()]
    if isinstance(raw, str):
        try:
            return [r for r in ast.literal_eval(raw) if r and str(r).strip()]
        except Exception:
            return [raw.strip()] if raw.strip() else []
    return []

# BUG: retain_start_idx stride used to be MAX_REPHRASES+1 even when rephrases
# were disabled, halving retain-sample diversity.  Use stride=1 when off.
retain_stride = (MAX_REPHRASES + 1) if USE_REPHRASES else 1

n_reph = 0
total_primary  = len(forget_dataset) * UNLEARN_STEPS
total_rephrase = len(forget_dataset) * MAX_REPHRASES * REPHRASE_STEPS if USE_REPHRASES else 0
print(f'[Phase 1] {len(forget_dataset)} samples x {UNLEARN_STEPS} steps  '
      f'(stride={retain_stride})')
print(f'  Est. Phase-1 updates: {total_primary:,} primary + '
      f'{total_rephrase:,} rephrase = {total_primary + total_rephrase:,} total')

for i, fs in enumerate(forget_dataset):
    maat.unlearn_step(
        forget_question  = fs['question'],
        forget_answer    = fs['answer'],
        retain_pool      = retain_list,
        retain_start_idx = i * retain_stride,
        tokenizer        = tokenizer,
        steps            = UNLEARN_STEPS,
    )
    if USE_REPHRASES:
        for j, rq in enumerate(get_rephrases(fs)[:MAX_REPHRASES]):
            maat.unlearn_step(
                forget_question  = rq,
                forget_answer    = fs['answer'],
                retain_pool      = retain_list,
                retain_start_idx = i * retain_stride + j + 1,
                tokenizer        = tokenizer,
                steps            = REPHRASE_STEPS,
            )
            n_reph += 1
    if (i + 1) % 50 == 0:
        torch.cuda.empty_cache()
        print(f'  [{i+1}/{len(forget_dataset)}]  nan_skips: {maat._nan_skip_count}  '
              f'rephrases: {n_reph}')

print(f'Phase 1 complete.  NaN skips: {maat._nan_skip_count}  '
      f'Rephrase variants: {n_reph}\n')

# ── Weight health check after Phase 1 ────────────────────────────────────
healthy_p1, _, _ = maat.check_weights_healthy('after Phase 1')
if not healthy_p1:
    n_restored = maat.recover_nan_weights()
    maat.check_weights_healthy('after NaN recovery')

# ── Phase 2a: MLP SVD pruning ─────────────────────────────────────────────
if DO_SVD_PRUNE:
    maat.svd_prune(forget_dataset, tokenizer, n_score_samples=SVD_SCORE_SAMPLES)
    maat.check_weights_healthy('after Phase 2a')

# ── Phase 2b: disabled ────────────────────────────────────────────────────
if DO_ATTN_PRUNE:
    maat.attn_micro_prune(forget_dataset, tokenizer, n_score_samples=20)

# ── Phase 2.5A: compute forget task vectors ───────────────────────────────
if DO_TASK_VEC:
    maat.compute_forget_task_vector(forget_dataset, tokenizer,
                                    n_score_samples=TV_SCORE_SAMPLES)

# ── Phase 2.5B: apply task vector negation ────────────────────────────────
if DO_TASK_VEC:
    maat.apply_task_vector_negation(alpha=TV_ALPHA)
    maat.check_weights_healthy('after Phase 2.5B')

# ── Spot-check coherence after Phase 2 + 2.5 ─────────────────────────────
model.eval()
_dev = get_input_device(model)
print('\n[Spot-check] Coherence after Phase 2+2.5:')
any_broken = severe_broken = 0
for _i in range(4):
    _s = retain_list[_i]
    _e = tokenizer(format_prompt_eval(_s['question']),
                   return_tensors='pt', truncation=True, max_length=256,
                   add_special_tokens=False)
    _e = {k: v.to(_dev) for k, v in _e.items()}
    _e['token_type_ids'] = torch.zeros_like(_e['input_ids'])
    with torch.no_grad():
        _g = model.generate(
            **_e, max_new_tokens=50, do_sample=False,
            eos_token_id=_eos_ids, pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.3,
        )
    _r = tokenizer.decode(_g[0][_e['input_ids'].shape[1]:],
                          skip_special_tokens=True).strip()
    _moderate = is_output_broken(_r, severe_only=False)
    _severe   = is_output_broken(_r, severe_only=True)
    if _moderate: any_broken += 1
    if _severe:   severe_broken += 1
    mark = 'SEVERE' if _severe else ('MODERATE' if _moderate else 'OK')
    _diag = f'len={len(_r)}'
    if not _r or len(_r.strip()) < 5:
        _diag += '  EMPTY/SHORT'
    else:
        _alnum = sum(c.isalnum() or c in " .,!?-" for c in _r) / max(1, len(_r))
        _diag += f'  alnum={_alnum:.0%}'
    print(f'  [{mark}] {_diag}')
    print(f'         Q: {_s["question"][:80]}')
    print(f'         A: {_r[:120]}\n')

if severe_broken >= 3:
    print(f'WARNING: {severe_broken}/4 samples severely broken after Phase 2+2.5.')
    print('Phase 3 repair will recover -- proceeding regardless.')
elif any_broken >= 2:
    print(f'WARNING: {any_broken}/4 moderate damage -- Phase 3 will recover.')
else:
    print('Spot-check passed cleanly.')

# ── Phase 3: hybrid repair ────────────────────────────────────────────────
if DO_RETAIN_REPAIR:
    maat.retain_repair(
        retain_dataset, tokenizer,
        forget_dataset     = forget_dataset,
        n_steps            = REPAIR_STEPS,
        repair_lr          = REPAIR_LR,
        kl_weight          = KL_WEIGHT,
        hs_weight          = HS_WEIGHT,
        forget_reg_weight  = FORGET_REG_WEIGHT,
        tv_weight          = TV_WEIGHT,
        repair_kl_temp     = REPAIR_KL_TEMP,
        final_lr           = FINAL_LR,
    )

maat.check_weights_healthy('after Phase 3')

if os.path.exists(UNLEARNED_ADAPTER): shutil.rmtree(UNLEARNED_ADAPTER)
model.save_pretrained(UNLEARNED_ADAPTER)
tokenizer.save_pretrained(UNLEARNED_ADAPTER)
print(f'\nUnlearned adapter saved -> {UNLEARNED_ADAPTER}')
print(f'Total Phase-1 NaN skips: {maat._nan_skip_count}')


## 6 · Merge Adapter → Full Model

In [ ]:
import gc, shutil, os
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME        = 'google/gemma-3-4b-it'
UNLEARNED_ADAPTER = '/kaggle/working/lora_adapter_unlearned'
MERGED_MODEL_PATH = '/kaggle/working/Gemma-3-4B-Unlearned-v9'

try:
    del model, maat, base_model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

print('Merging unlearned adapter -> full Gemma model (bfloat16)...')
tokenizer = AutoTokenizer.from_pretrained(UNLEARNED_ADAPTER)

# BUG-1 FIX: load for merge in bfloat16 to match training dtype
base_for_merge = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto'  # BUG-1 FIX
)
_cur_vocab = base_for_merge.get_input_embeddings().weight.shape[0]
if len(tokenizer) != _cur_vocab:
    base_for_merge.resize_token_embeddings(len(tokenizer))
    print(f'Resized embeddings: {_cur_vocab} -> {len(tokenizer)}')

merged = PeftModel.from_pretrained(
    base_for_merge, UNLEARNED_ADAPTER).merge_and_unload()

if os.path.exists(MERGED_MODEL_PATH): shutil.rmtree(MERGED_MODEL_PATH)
merged.save_pretrained(MERGED_MODEL_PATH)
tokenizer.save_pretrained(MERGED_MODEL_PATH)
print(f'Merged model -> {MERGED_MODEL_PATH}')

del merged, base_for_merge
gc.collect(); torch.cuda.empty_cache()


## 7 · Qualitative Evaluation

In [ ]:
import json, os, gc
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM

MERGED_MODEL_PATH = '/kaggle/working/Gemma-3-4B-Unlearned-v9'
DATA_DIR          = '/kaggle/working/data_splits'
OUT_DIR           = '/kaggle/working/eval_inputs'
MAX_NEW_TOKENS    = 100

os.makedirs(OUT_DIR, exist_ok=True)
gc.collect(); torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<pad>'})

model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH, torch_dtype=torch.bfloat16, device_map='auto'
)
model.eval()
input_device = get_input_device(model)
torch.manual_seed(42)

eos_ids = list(set(filter(lambda x: x is not None and x >= 0, [
    tokenizer.convert_tokens_to_ids('<end_of_turn>'),
    tokenizer.eos_token_id,
])))
print(f'EOS ids: {eos_ids}')


def generate_answer(question: str) -> str:
    prompt  = format_prompt_eval(question)
    inputs  = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=512,
                        add_special_tokens=False).to(input_device)
    inputs['token_type_ids'] = torch.zeros_like(inputs['input_ids'])
    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens     = MAX_NEW_TOKENS,
            do_sample          = False,
            repetition_penalty = 1.3,
            eos_token_id       = eos_ids,
            pad_token_id       = tokenizer.pad_token_id,
        )
    new_ids = gen[0][inputs['input_ids'].shape[1]:]
    return clean_output(tokenizer.decode(new_ids, skip_special_tokens=True).strip())


def collect_answers(dataset, label_feature, split_name: str) -> list:
    records = []
    print(f'[{split_name}] {len(dataset)} samples...')
    for i, sample in enumerate(dataset):
        records.append({
            'split'        : split_name,
            'idx'          : i,
            'label'        : label_feature.int2str(sample['label']),
            'question'     : sample['question'],
            'ground_truth' : sample['answer'],
            'model_answer' : generate_answer(sample['question']),
        })
        if (i + 1) % 50 == 0:
            torch.cuda.empty_cache()
            print(f'  [{i+1}/{len(dataset)}]')
    return records


forget_dataset = load_from_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset = load_from_disk(f'{DATA_DIR}/retain_dataset')
label_feature  = forget_dataset.features['label']

forget_records = collect_answers(forget_dataset, label_feature, 'forget')
retain_records = collect_answers(retain_dataset, label_feature, 'retain')
all_records    = forget_records + retain_records

for fname, records in [
    ('forget_answers.json', forget_records),
    ('retain_answers.json', retain_records),
    ('all_answers.json',    all_records),
]:
    with open(os.path.join(OUT_DIR, fname), 'w') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

print(f'\nSaved to {OUT_DIR}/')
for fname in ['forget_answers.json', 'retain_answers.json', 'all_answers.json']:
    p = os.path.join(OUT_DIR, fname)
    print(f'  {p}  ({os.path.getsize(p)/1024:.1f} KB)')
print('\n-- Sample forget --')
print(json.dumps(forget_records[0], indent=2, ensure_ascii=False))
print('\n-- Sample retain --')
print(json.dumps(retain_records[0], indent=2, ensure_ascii=False))


## 8 · ROUGE Evaluation

In [ ]:
import evaluate, json, os, gc, re, string
import torch
from tqdm import tqdm
from datasets import load_from_disk
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

MERGED_MODEL_PATH = '/kaggle/working/Gemma-3-4B-Unlearned-v9'
DATA_DIR          = '/kaggle/working/data_splits'
OUT_DIR           = '/kaggle/working/eval_inputs'
ROUGE_OUT         = f'{OUT_DIR}/rouge_scores.json'

rouge_metric = evaluate.load('rouge')

try:
    _ = model
    print('Using model already in memory.')
except NameError:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '<pad>'})
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_PATH, torch_dtype=torch.bfloat16, device_map='auto')
    model.eval()
    print('Model loaded from disk.')

eos_ids = list(set(filter(lambda x: x is not None and x >= 0, [
    tokenizer.convert_tokens_to_ids('<end_of_turn>'),
    tokenizer.eos_token_id,
])))
input_device = next(model.parameters()).device


def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()


def prepare_for_rouge(text: str) -> str:
    text = normalize_text(text)
    return '\n'.join(sent_tokenize(text)) if text else text


def _generate(question: str) -> str:
    prompt  = format_prompt_eval(question)
    inputs  = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=512,
                        add_special_tokens=False).to(input_device)
    inputs['token_type_ids'] = torch.zeros_like(inputs['input_ids'])
    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=100, do_sample=False,
            repetition_penalty=1.3,
            eos_token_id=eos_ids, pad_token_id=tokenizer.pad_token_id,
        )
    raw = tokenizer.decode(gen[0][inputs['input_ids'].shape[1]:],
                           skip_special_tokens=True).strip()
    return clean_output(raw)


def compute_rouge(dataset_path: str, split_name: str) -> dict:
    print(f'\n-- {split_name} -------------------------')
    ds = load_from_disk(dataset_path)
    preds, refs = [], []
    for sample in tqdm(ds, desc=f'{split_name} generation'):
        preds.append(prepare_for_rouge(_generate(sample['question'])))
        refs.append(prepare_for_rouge(sample['answer']))
    scores = rouge_metric.compute(predictions=preds, references=refs,
                                  use_stemmer=True)
    print(f'  ROUGE-1 : {scores["rouge1"]:.4f}')
    print(f'  ROUGE-2 : {scores["rouge2"]:.4f}')
    print(f'  ROUGE-L : {scores["rougeL"]:.4f}')
    return {'split': split_name, **scores}


forget_scores = compute_rouge(f'{DATA_DIR}/forget_dataset', 'Forget (unlearned)')
retain_scores = compute_rouge(f'{DATA_DIR}/retain_dataset', 'Retain (preserved)')

summary = {'forget': forget_scores, 'retain': retain_scores}
os.makedirs(OUT_DIR, exist_ok=True)
with open(ROUGE_OUT, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nROUGE scores saved -> {ROUGE_OUT}')

print('\n== Summary (Gemma-3-4B, 500+500 samples) =======================')
print(f'{"Metric":<12}  {"Forget":>10}  {"Retain":>10}')
print(f'{"-"*36}')
for m in ['rouge1', 'rouge2', 'rougeL']:
    print(f'{m:<12}  {forget_scores[m]:>10.4f}  {retain_scores[m]:>10.4f}')
print()
print('Llama-3.2-3B baseline (100+100 samples):')
print(f'{"rouge1":<12}  {"0.4683":>10}  {"0.5747":>10}')
print(f'{"rouge2":<12}  {"0.2817":>10}  {"0.3632":>10}')
print(f'{"rougeL":<12}  {"0.4580":>10}  {"0.5695":>10}')
print()
print('Target: Forget ROUGE << Llama baseline  (unlearning working)')
print('        Retain ROUGE ~= Llama baseline  (retention preserved)')
